# Experiment: Cross-Method Tiny-p Study

Objective:
- Compare `iid`, `mcmc_is`, and `samc` on fixed exact scenarios.
- Track estimates and diagnostics at intermediate estimation points, not just at the final budget.

In [ ]:
from __future__ import annotations

from dataclasses import replace
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "perm_pval").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing perm_pval/ and results/.")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.environ.setdefault("MPLCONFIGDIR", str(project_root / ".matplotlib"))

from perm_pval.experiments.notebook_studies import (
    BetaSweepStudyConfig,
    CrossMethodStudyConfig,
    DEFAULT_MCMC_OBJECTIVE_GRID_Q_MULTIPLIERS,
    DEFAULT_MCMC_OBJECTIVE_GRID_SWAP_COUNTS,
    MCMCWorkflowConfig,
    MCMC_OBJECTIVE_GRID_REALISTIC_OBJECTIVES,
    SAMCWorkflowConfig,
    build_beta_workflow,
    create_timestamped_run_dir,
    load_beta_sweep_saved_output,
    load_cross_method_saved_output,
    load_mcmc_objective_grid_saved_output,
    load_selected_scenarios,
    run_mcmc_objective_grid_study,
    save_mcmc_objective_grid_outputs,
    regenerate_beta_sweep_plots_from_saved,
    regenerate_cross_method_plots_from_saved,
    run_beta_checkpoint_study,
    run_cross_method_study,
    save_beta_sweep_outputs,
    save_cross_method_outputs,
    summarize_records,
)

pd.set_option("display.max_columns", 100)
project_root

## Configuration

`ESTIMATION_POINTS` controls the intermediate checkpoints.  
The largest checkpoint is used for the main boxplots; all checkpoints are used for convergence diagnostics.  
In the cross-method notebook these are total budgets. For MCMC-IS, a fixed beta-selection budget is deducted first, and the production chain uses the remaining budget.

In [ ]:
FAST_MODE = False
SAVE_OUTPUTS = True

CATALOG_PATH = project_root / "results" / "exact_scenarios" / "v1" / "catalog.json"
OUTPUT_ROOT = project_root / "results" / "cross_method_notebook"

SCENARIO_GROUP = "core_claim"
SCENARIO_KEYS_OVERRIDE = [
    "hypergeom_1e7",
    "gwas_additive_score_n40",
    "rank_sum_dp_n40",
    "linear_stat_dp_n40",
    "bruteforce_welch_nonextreme_n22",
]

ESTIMATION_POINTS = (750_000, 1_000_000, 2_500_000, 5_000_000, 7_500_000, 10_000_000, 15_000_000) if not FAST_MODE else (50_000, 100_000, 200_000)
N_REPEATS = 10 if not FAST_MODE else 2
N_JOBS = min(N_REPEATS, os.cpu_count() or 1)
MIN_TAIL_STATES = 2
BASE_SEED = 12_345
MCMC_LOCAL_SCAN_OBJECTIVE = "varhat_qmatch_soft"
MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND = {
    "small": 1,
    "medium": 1,
    "large": 2,
}
MCMC_LOCAL_SCAN_Q_MULTIPLIERS = (0.001, 0.005, 0.01, 0.05, 0.10, 0.15, 0.25, 0.33, 1.0)

cross_cfg = CrossMethodStudyConfig(
    estimation_points=ESTIMATION_POINTS,
    repeats=N_REPEATS,
    base_seed=BASE_SEED,
    iid_density_samples=150_000 if not FAST_MODE else 10_000,
    min_tail_states=MIN_TAIL_STATES,
    n_jobs=N_JOBS,
)
base_mcmc_cfg = MCMCWorkflowConfig(
    pilot_samples=25_000 if not FAST_MODE else 1_000,
    tune_steps=3_000 if not FAST_MODE else 1_000,
    local_scan_screen_total_steps=14_000 if not FAST_MODE else 1_000,
    local_scan_total_steps=32_000 if not FAST_MODE else 6_000,
    chains=2,
    thin=1,
    estimate_variance=True,
    proposal_size=1,
    local_scan_objective=MCMC_LOCAL_SCAN_OBJECTIVE,
    local_scan_q_multipliers=MCMC_LOCAL_SCAN_Q_MULTIPLIERS,
)
samc_cfg = SAMCWorkflowConfig(
    n_bins=100,
    t0=1_000.0,
    trace_every=200 if not FAST_MODE else 50,
    lambda_min_pilot=10_000 if not FAST_MODE else 500,
    proposal_size=0.1,
)

def mcmc_proposal_size_for_scenario(scenario):
    band = str(scenario.portfolio.get("sample_size_band", "medium"))
    return int(MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND.get(band, 1))


def mcmc_cfg_for_scenario(scenario):
    proposal_size = int(mcmc_proposal_size_for_scenario(scenario))
    return replace(
        base_mcmc_cfg,
        proposal_size=proposal_size,
        local_scan_swap_counts=(proposal_size,),
    )

print(json.dumps({
    "FAST_MODE": FAST_MODE,
    "SCENARIO_GROUP": SCENARIO_GROUP,
    "SCENARIO_KEYS_OVERRIDE": SCENARIO_KEYS_OVERRIDE,
    "ESTIMATION_POINTS": ESTIMATION_POINTS,
    "N_REPEATS": N_REPEATS,
    "N_JOBS": N_JOBS,
    "SAVE_OUTPUTS": SAVE_OUTPUTS,
    "MCMC_LOCAL_SCAN_OBJECTIVE": MCMC_LOCAL_SCAN_OBJECTIVE,
    "MCMC_LOCAL_SCAN_Q_MULTIPLIERS": MCMC_LOCAL_SCAN_Q_MULTIPLIERS,
    "MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND": MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND,
}, indent=2))

## Load Scenarios

In [ ]:
scenarios = load_selected_scenarios(
    catalog_path=CATALOG_PATH,
    scenario_keys=SCENARIO_KEYS_OVERRIDE,
    portfolio_group=None if SCENARIO_KEYS_OVERRIDE is not None else SCENARIO_GROUP,
    min_tail_states=MIN_TAIL_STATES,
)

run_dir = create_timestamped_run_dir(OUTPUT_ROOT, "cross_method") if SAVE_OUTPUTS else None

pd.DataFrame([
    {
        "scenario": s.key,
        "exact_p": s.exact_p,
        "tail_hits": s.exact_tail_hits,
        "n_perm": s.exact_n_perm,
        "exact_method": s.exact_method,
        "family": s.portfolio.get("family"),
        "rarity_band": s.portfolio.get("rarity_band"),
        "difficulty": s.portfolio.get("expected_difficulty"),
        "groups": ",".join(s.portfolio.get("groups", [])),
    }
    for s in scenarios
])

## Run Cross-Method Study

For each scenario:
- build one MCMC-IS beta workflow,
- evaluate all methods at every checkpoint in `ESTIMATION_POINTS`,
- save max-budget and convergence plots.

In [ ]:
cross_results = {}

for scenario in scenarios:
    scenario_mcmc_cfg = mcmc_cfg_for_scenario(scenario)
    print(f"Running {scenario.key} | exact p={scenario.exact_p:.3e}")
    print(json.dumps({
        "scenario": scenario.key,
        "sample_size_band": scenario.portfolio.get("sample_size_band"),
        "mcmc_proposal_size": scenario_mcmc_cfg.proposal_size,
        "mcmc_local_scan_swap_counts": scenario_mcmc_cfg.local_scan_swap_counts,
        "mcmc_local_scan_objective": scenario_mcmc_cfg.local_scan_objective,
    }, indent=2))
    study = run_cross_method_study(
        scenario,
        cross_cfg=cross_cfg,
        mcmc_cfg=scenario_mcmc_cfg,
        samc_cfg=samc_cfg,
    )
    cross_results[scenario.key] = study

    if SAVE_OUTPUTS and run_dir is not None:
        save_cross_method_outputs(
            scenario,
            study,
            output_dir=run_dir / scenario.key,
            cross_cfg=cross_cfg,
            mcmc_cfg=scenario_mcmc_cfg,
            samc_cfg=samc_cfg,
        )

    print(json.dumps({
        "scenario": scenario.key,
        "mcmc_beta_selection_budget": study["mcmc_beta_selection_budget"],
        "beta_used": study["beta_workflow"]["beta_used"],
    }, indent=2))
    summary_df = pd.DataFrame(study["summary"]).sort_values(["checkpoint", "method"])
    display(summary_df[[
        "checkpoint",
        "method",
        "mean_estimate",
        "rmse",
        "mean_variance_estimate",
        "mean_eval_incl_tuning",
        "mean_q_tilt_tail_share",
        "mean_ess",
        "mean_zero_rate",
        "mean_samc_max_rel_freq_error",
    ]])

family_rows = []
for scenario in scenarios:
    meta = scenario.portfolio
    for row in cross_results[scenario.key]["summary"]:
        family_rows.append({
            "scenario": scenario.key,
            "family": meta.get("family"),
            "rarity_band": meta.get("rarity_band"),
            "difficulty": meta.get("expected_difficulty"),
            **row,
        })

family_df = pd.DataFrame(family_rows)
display(
    family_df.groupby(["family", "rarity_band", "method", "checkpoint"], as_index=False)
    .agg(
        mean_rmse=("rmse", "mean"),
        mean_estimate=("mean_estimate", "mean"),
        mean_q_tilt_tail_share=("mean_q_tilt_tail_share", "mean"),
        mean_ess=("mean_ess", "mean"),
    )
    .sort_values(["family", "rarity_band", "checkpoint", "method"])
)

## Review Saved Figures

In [ ]:
if SAVE_OUTPUTS and run_dir is not None:
    print(f"Saved outputs under: {run_dir}")
    for scenario in scenarios:
        scenario_dir = run_dir / scenario.key
        print(f"\n{scenario.key}")
        display(Image(filename=str(scenario_dir / "cross_method_max_budget.png")))
        display(Image(filename=str(scenario_dir / "cross_method_convergence.png")))
        display(Image(filename=str(scenario_dir / "cross_method_diagnostics.png")))
        display(Image(filename=str(scenario_dir / "iid_density.png")))
else:
    print("SAVE_OUTPUTS=False, so no saved figures to display.")

## Reload Saved Results Without Rerunning

In [ ]:
# RELOAD_SCENARIO_DIR = None
# # Example:
# # RELOAD_SCENARIO_DIR = project_root / "results" / "cross_method_notebook" / "20260306_120000_cross_method" / "gwas_additive_score_n40"

# if RELOAD_SCENARIO_DIR is not None:
#     saved = load_cross_method_saved_output(RELOAD_SCENARIO_DIR)
#     print(json.dumps({
#         "scenario": saved["metadata"]["scenario"],
#         "exact_p": saved["metadata"]["exact_p"],
#         "mcmc_beta_selection_budget": saved["metadata"]["beta_workflow"]["beta_selection_eval_total"],
#     }, indent=2))
#     regen = regenerate_cross_method_plots_from_saved(RELOAD_SCENARIO_DIR)
#     for name, path in regen.items():
#         print(name, path)
#         display(Image(filename=str(path)))
# else:
#     print("Set RELOAD_SCENARIO_DIR to a saved scenario directory to regenerate plots from disk only.")